In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
%matplotlib inline

# dropped columns
1. title
2. tagline
3. production_companies
4. production_countries
5. budget
6. revenue
7. homepage
8. status
9. original_language
10. spoken_language
11. popularity
12. genres
13. keywords
14. cast
15. crew
16. overview

In [2]:
# datas
df = pd.read_csv("tmdb_movies.csv")
df_credits = pd.read_csv("tmdb_credits.csv")

In [3]:
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [4]:
df_credits.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')

In [5]:
df[['original_title', 'title']]

,original_title,title
0,Avatar,Avatar
1,Pirates of the Caribbean: At World's End,Pirates of the Caribbean: At World's End
2,Spectre,Spectre
3,The Dark Knight Rises,The Dark Knight Rises
4,John Carter,John Carter
...,...,...
4798,El Mariachi,El Mariachi
4799,Newlyweds,Newlyweds
4800,"Signed, Sealed, Delivered","Signed, Sealed, Delivered"
4801,Shanghai Calling,Shanghai Calling


## Observation: In movies dataset(df) original title and title have same values so lets drop title and rename 'original title' as 'title'

In [6]:
# dropping title
df.drop('title', axis = 1, inplace = True)
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count'],
      dtype='object')

In [7]:
# rename og_title as title
df.rename(columns ={'original_title':'title'}, inplace = True)

In [8]:
df['id'].isnull().sum()

np.int64(0)

In [9]:
df_credits['movie_id'].isnull().sum()

np.int64(0)

In [10]:
df_credits.rename(columns = {'movie_id':'id'}, inplace = True)

combine credits dataset with the movie dataset to make df as the main dataset. df already has id and title so add only casts and crew to it.

In [11]:
# merging df_credits to df
df = df.merge(df_credits[['id', 'cast', 'crew']], on='id', how='left')

In [12]:
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count',
       'cast', 'crew'],
      dtype='object')

In [13]:
#df.drop(columns = ['cast_y', 'crew_y'], inplace = True)
#df.rename(columns = {'cast_x':'cast','crew_x':'crew'}, inplace = True)

In [14]:
# basic run
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   title                 4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4.803000e+03,4803.000000,4803.000000,4.803000e+03,4801.000000,4803.000000,4803.000000
mean,2.904504e+07,57165.484281,21.492301,8.226064e+07,106.875859,6.092172,690.217989
std,4.072239e+07,88694.614033,31.816650,1.628571e+08,22.611935,1.194612,1234.585891
min,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,7.900000e+05,9014.500000,4.668070,0.000000e+00,94.000000,5.600000,54.000000
50%,1.500000e+07,14629.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000
75%,4.000000e+07,58610.500000,28.313505,9.291719e+07,118.000000,6.800000,737.000000
max,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000


In [15]:
# missing values
[features for features in df.columns if df[features].isnull().sum()>0]

['homepage', 'overview', 'release_date', 'runtime', 'tagline']

Now extract release year from release date, combine overview with tagline, try to fill the missing values

## FEATURE ENGINEERING

In [16]:
# extract release year from date (yyyy - mm- dd) -----> (yyyy)

df['release_date'] = pd.to_datetime(df['release_date'], errors = 'coerce') # proper format coversion
df['release_date'].head()
df['release_date'] = df['release_date'].dt.year
df.rename(columns = {'release_date':'release_year'}, inplace = True)


In [17]:
# fill the release year
df['release_year'] = df['release_year'].fillna(0).astype(int)

In [18]:
# combine tagline with overview

df['tagline'] = df['tagline'].fillna(' ')
df['overview'] = df['overview'] + " " + df['tagline']

# drop tagline
df.drop(columns = ['tagline'], inplace = True)

In [19]:
# fill overview
df['overview'] = df['overview'].fillna(' ')

In [20]:
# reduce overview length
df['overview'] = df['overview'].apply(
    lambda x: " ".join(x.split()[:50])
)

In [21]:
# drop budget, homepage, original language, revenue, production companies and countries, popularity
df.drop(columns = ['budget', 'homepage', 'original_language', 
                    'popularity', 'production_companies', 
                    'production_countries', 'revenue', 'spoken_languages'], axis = 1, inplace = True)

In [22]:
df.columns

Index(['genres', 'id', 'keywords', 'title', 'overview', 'release_year',
       'runtime', 'status', 'vote_average', 'vote_count', 'cast', 'crew'],
      dtype='object')

In [23]:
# fill runtime
df['runtime'] = df['runtime'].fillna(df['runtime'].mean()).astype(int)

In [24]:
# keep only released movies
df = df[df['status'] == 'Released'].copy()

# now drop status
df.drop(columns = ['status'], axis = 1, inplace = True)

In [25]:
df[['genres', 'keywords', 'cast', 'crew']].head()

,genres,keywords,cast,crew
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


### Clean Cast and Crew

In [26]:
# CAST
def extract_cast(text):
    cast_list = []
    for i, item in enumerate(ast.literal_eval(text)):
        if i < 3:
            cast_list.append(item['name'])
    return cast_list

df['cast'] = df['cast'].apply(extract_cast)

In [27]:
# CREW
def extract_director(text):
    for item in ast.literal_eval(text):
        if item['job'] == 'Director':
            return [item['name']]
    return []

df['crew'] = df['crew'].apply(extract_director)

In [28]:
# close the gaps between names
df['cast'] = df['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
df['crew'] = df['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [29]:
# convert list to string
df['cast'] = df['cast'].apply(lambda x: " ".join(x))
df['crew'] = df['crew'].apply(lambda x: " ".join(x))

### clean genres and keywords

In [30]:
def extract_names(text):
    names = []
    for item in ast.literal_eval(text):
        names.append(item['name'])
    return names

In [31]:
df['genres'] = df['genres'].apply(extract_names)
df['keywords'] = df['keywords'].apply(extract_names)

In [32]:
# remove spaces
df['genres'] = df['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])

In [33]:
# list into string
df['genres'] = df['genres'].apply(lambda x: " ".join(x))
df['keywords'] = df['keywords'].apply(lambda x: " ".join(x))

In [34]:
# lowercase
df['genres'] = df['genres'].str.lower()
df['keywords'] = df['keywords'].str.lower()

In [35]:
df['vote_count'].max()
#df['vote_count'].min()

13752

In [36]:
df['vote_average'].max()
df['vote_average'].min()

0.0

In [37]:
# remove movies with 0 votes
df = df[df['vote_count'] > 0].copy()

In [38]:
df['vote_count'].describe()

count     4734.000000
mean       700.269328
std       1240.721060
min          1.000000
25%         59.000000
50%        242.500000
75%        754.000000
max      13752.000000
Name: vote_count, dtype: float64

In [39]:
df = df[df['vote_count'] >= 60].copy()

In [40]:
df.reset_index(drop=True, inplace=True)

In [41]:
df.shape

(3537, 11)

In [42]:
df.columns

Index(['genres', 'id', 'keywords', 'title', 'overview', 'release_year',
       'runtime', 'vote_average', 'vote_count', 'cast', 'crew'],
      dtype='object')

### stemming

In [43]:
ps = PorterStemmer()

def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

In [44]:
df['overview'] = df['overview'].apply(stem)

# COMBINING EVERYTHING

In [45]:
df['tags'] = (
    df['genres'] + " " +
    df['genres'] + " " +

    df['keywords'] + " " +
    df['keywords'] + " " +
    df['keywords'] + " " +

    df['cast'] + " " +
    df['cast'] + " " +

    df['crew'] + " " +
    df['crew'] + " " +

    df['title'] + " " +
    df['title'] + " " +
    df['title'] + " " +

    df['overview']
)

In [46]:
# now strip and remove the symbols
df['tags'] = df['tags'].str.lower()
df['tags'] = df['tags'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)

In [47]:
# df.drop(columns = ['genres', 'keywords', 'cast', 'crew', 'overview'], axis = 1, inplace = True)

In [48]:
df['tags'] = df['tags'].str.lower()

In [49]:
# Removing special characters from tags
df['tags'] = df['tags'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)

In [50]:
print(df['tags'].iloc[0])

action adventure fantasy sciencefiction action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver samworthington zoesaldana sigourneyweaver jamescameron jamescameron avatar avatar avatar in the 22nd century a parapleg marin is dispatch to the moon pandora on a uniqu mission but becom torn between follow order and protect an alien civilization enter the world of pandora


#### Vectorization / removing stopwords

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1,2)
)

vectors = tfidf.fit_transform(df['tags']).toarray()

### Link creation

In [52]:
df['tmdb_link'] = df['id'].apply(
    lambda x: "https://www.themoviedb.org/movie/" + str(x)
)

In [53]:
df['tmdb_link'].head()

0     https://www.themoviedb.org/movie/19995
1       https://www.themoviedb.org/movie/285
2    https://www.themoviedb.org/movie/206647
3     https://www.themoviedb.org/movie/49026
4     https://www.themoviedb.org/movie/49529
Name: tmdb_link, dtype: object

# COSINE SIMILARITY

In [54]:
sim = cosine_similarity(vectors)
sim.shape

(3537, 3537)

In [55]:
df["original_title"] = df["title"]
df["title"] = df["title"].str.lower()

### RECOMMENDATION FUNCTION

In [56]:
def recommend(movie):

    movie = movie.lower()

     # Check if movie exists
    if movie not in df['title'].values:
        print("Movie not found")
        return
    print("You might enjoy these....\n")

    movie_index = df[df['title'] == movie].index[0]
    
    distances = sim[movie_index]
    
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movies_list:
        print("Movie:", df.iloc[i[0]].original_title)
        print("Link:", df.iloc[i[0]].tmdb_link)
        print("Year:", df.iloc[i[0]].release_year)
        print()

In [68]:
df.columns

Index(['genres', 'id', 'keywords', 'title', 'overview', 'release_year',
       'runtime', 'vote_average', 'vote_count', 'cast', 'crew', 'tags',
       'tmdb_link', 'original_title'],
      dtype='object')

### User Input

In [70]:
title_input = input("Enter the movie title: ")
title_input = title_input.lower()
print("_____________________________________________________________________________________________________________________")
print()
recommend(title_input)

Enter the movie title:  inception


_____________________________________________________________________________________________________________________

You might enjoy these....

Movie: Heist
Link: https://www.themoviedb.org/movie/336004
Year: 2015

Movie: Takers
Link: https://www.themoviedb.org/movie/22907
Year: 2010

Movie: Солярис
Link: https://www.themoviedb.org/movie/593
Year: 1972

Movie: Triple 9
Link: https://www.themoviedb.org/movie/146198
Year: 2016

Movie: Transporter 2
Link: https://www.themoviedb.org/movie/9335
Year: 2005

